<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/10_PGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm
import statsmodels.api as sm

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal
from matplotlib.ticker import AutoMinorLocator, LogLocator

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = ''

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar CSV

In [ ]:
df_ = pd.read_csv('/content/SeismicUP/Outputs/20260718_Eventos_VMM_radio150km_v3_ajustado.csv',
                  sep=';',
                  decimal=',')


df_ = df_[df_['validas'] == 1].copy()
df_.reset_index(drop=True, inplace=True)

new_row = pd.DataFrame({
    'Región': ['BRJC'],
    'Latitud': [6.97],
    'Longitud': [-73.74],
    'Profundidad [km]': [108]
})

df_ = pd.concat([df_, new_row], ignore_index=True)

df_.info()

# Geodataframe

In [ ]:
# Incluir las coordenadsa para georfrenciar la estacion

In [ ]:
gdf_unido = gpd.GeoDataFrame(
    df_, geometry=gpd.points_from_xy(df_.Longitud, df_.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_unido.plot()

In [ ]:
gdf_unido = gdf_unido.to_crs(9377)

gdf_unido.plot()

## Obtener X y Y

In [ ]:
gdf_unido.get_coordinates()

In [ ]:
df_['X [km]'] = gdf_unido.get_coordinates()['x'] / 1000
df_['Y [km]'] = gdf_unido.get_coordinates()['y'] / 1000

df_.tail()

In [ ]:
df_.info()

# DF PGA

In [ ]:
df_.columns

In [ ]:
df_PGA = df_[['X [km]', 'Error Longitud [km]', 'Y [km]', 'Error Latitud [km]',
       'Profundidad [km]', 'Error Profundidad [km]', 'Magnitud',
       'Tipo Magnitud', 'PGA EW (cm/s^2)', 'PGA NS (cm/s^2)',
       'PGA Z (cm/s^2)', ]].copy()
df_PGA = df_PGA.iloc[:-1]

df_BRJC = df_.iloc[-1].copy()
df_BRJC = df_BRJC[['X [km]', 'Y [km]', 'Profundidad [km]']]

print(df_BRJC, '\n')

df_PGA

# Plots

### Histograma errores

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Y [km]',
                          ax_h= ax[0,0],
                          dist_min= df_PGA['Y [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'X [km]',
                          ax_h= ax[0,1],
                          dist_min= df_PGA['X [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0,2],
                          dist_min= df_PGA['Profundidad [km]'].min(),
                          step_= 2)

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Error Latitud [km]',
                          ax_h= ax[1,0],
                          dist_min= 0,
                          step_= 1)

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Error Longitud [km]',
                          ax_h= ax[1,1],
                          dist_min= 0,
                          step_= 1)

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1,2],
                          dist_min= 0,
                          step_= 1)

ax[0,2].set_xlabel('Depth [km]')
ax[1,0].set_xlabel('Latitude Error [km]')
ax[1,1].set_xlabel('Longitude Error [km]')
ax[1,2].set_xlabel('Depth Error [km]')

# sup.plots.save_plot(path_save_figuras = path_save_figures,
#                     file_name = 'hitograma_errores_df_filtrado',
#                     formato = 'png',
#                     dpi_ = 300)

plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4,4))

sup.plots.histograma_plot(df= df_PGA,
                          columna= 'Magnitud',
                          ax_h= ax,
                          dist_min=0,
                          step_= 0.5)

# sup.plots.histograma_plot(df= df_PGA,
#                           columna= 'Error magnitud',
#                           ax_h= ax[1],
#                           dist_min= 0,
#                           step_= 0.050)

ax.set_xlabel('Magnitud')
# ax[1].set_xlabel('Error magnitud')

# sup.plots.save_plot(path_save_figuras = path_save_figures,
#                     file_name = 'hitograma_errores_df_filtrado',
#                     formato = 'png',
#                     dpi_ = 300)

plt.show()

# Homogeneizar magnitudes

In [ ]:
# no todas son Mw ... se va usar la formula del SGC para pasar a Mw
# considerar para la de Campo Colorado usar mejor Ml
# que ya lo entrega así el SGC
df_PGA['Tipo Magnitud'].unique()

In [ ]:
# Casi todas dicen Ml ... para M no aparece una formula
# se va usar la misma de Ml
# Mw (2.9 <Ml <= 6.1) = 0.958*Ml + 0.1
# df_PGA['Mw'] = df_PGA['Magnitud'] * 0.958 + 0.1

# Para dejas Mw igual
# df_PGA['Mw'][df_PGA['Tipo magnitud'].str.contains('Mw')] = df_PGA['Magnitud'][df_PGA['Tipo magnitud'].str.contains('Mw')]
df_PGA['Ml'] = df_PGA['Magnitud'].copy()
df_PGA.head(10)

# Eliminar nulos

In [ ]:
df_PGA.dropna(inplace=True, subset=['PGA EW (cm/s^2)',
       'PGA NS (cm/s^2)', 'PGA Z (cm/s^2)'])
df_PGA.reset_index(drop=True, inplace=True)
df_PGA.info()

In [ ]:
# algunos errores estan vacios
df_PGA.describe().round(2)

In [ ]:
df_PGA_aumentado = df_PGA.copy()

In [ ]:
df_PGA_aumentado = df_PGA[['X [km]', 'Y [km]',
                           'Profundidad [km]', 'Ml',
                           'PGA EW (cm/s^2)',
                           'PGA NS (cm/s^2)',
                           'PGA Z (cm/s^2)']].copy()
df_PGA_aumentado.round(3)

### Histograma

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(12,4))

sup.plots.histograma_plot(df= df_PGA_aumentado,
                          columna= 'X [km]',
                          ax_h= ax[0],
                          dist_min= df_PGA_aumentado['X [km]'].min(),
                          step_= 20)

sup.plots.histograma_plot(df= df_PGA_aumentado,
                          columna= 'Y [km]',
                          ax_h= ax[1],
                          dist_min= df_PGA_aumentado['Y [km]'].min(),
                          step_= 20)


sup.plots.histograma_plot(df= df_PGA_aumentado,
                          columna= 'Profundidad [km]',
                          ax_h= ax[2],
                          dist_min= df_PGA_aumentado['Profundidad [km]'].min(),
                          step_= 2)

sup.plots.histograma_plot(df= df_PGA_aumentado,
                          columna= 'Ml',
                          ax_h= ax[3],
                          dist_min=0,
                          step_= 0.5)


ax[0].set_xlabel('X [km]')
ax[1].set_xlabel('Y [km]')
ax[2].set_xlabel('Depth [km]')
ax[3].set_xlabel('Ml')

# sup.plots.save_plot(path_save_figuras = path_save_figures,
#                     file_name = 'hitograma_errores_df_filtrado',
#                     formato = 'png',
#                     dpi_ = 300)

plt.show()

# Calculo parametros para PGA

In [ ]:
df_BRJC

In [ ]:
# Distancia eventos a la estacion BRJC
df_PGA_aumentado['Dist Epi [km]'] = np.sqrt(
    (df_PGA_aumentado['X [km]'] - df_BRJC['X [km]'])**2 +
    (df_PGA_aumentado['Y [km]'] - df_BRJC['Y [km]'])**2)

df_PGA_aumentado['Dist Hip [km]'] = np.sqrt(
    (df_PGA_aumentado['X [km]'] - df_BRJC['X [km]'])**2 +
    (df_PGA_aumentado['Y [km]'] - df_BRJC['Y [km]'])**2 +
    (df_PGA_aumentado['Profundidad [km]'] - df_BRJC['Profundidad [km]'])**2)

df_PGA_aumentado.describe().round(2)

In [ ]:
p1 = (0, 0)
p2 = (-1, -1)

angulo = np.arctan2(p2[0] - p1[0],
                    p2[1] - p1[1]) * 180 / np.pi

angulo = (angulo+ 360) % 360

angulo

In [ ]:
# Rumbo desde la estacion BRJC al evento
# de 0 a 360 grados
df_PGA_aumentado['Rumbo [°]'] = np.arctan2(df_PGA_aumentado['Y [km]'] - df_BRJC['Y [km]'],
                                           df_PGA_aumentado['X [km]'] - df_BRJC['X [km]']) * 180 / np.pi

df_PGA_aumentado['Rumbo [°]'] = (df_PGA_aumentado['Rumbo [°]'] + 360) % 360

df_PGA_aumentado.describe().round(2)

In [ ]:
df_PGA_aumentado['Seno'] = np.sin(np.deg2rad(df_PGA_aumentado['Rumbo [°]']))
df_PGA_aumentado['Coseno'] = np.cos(np.deg2rad(df_PGA_aumentado['Rumbo [°]']))

df_PGA_aumentado.describe().round(2)

In [ ]:
df_PGA_aumentado['logEpi'] = np.log10(df_PGA_aumentado['Dist Epi [km]'])
df_PGA_aumentado['logHip'] = np.log10(df_PGA_aumentado['Dist Hip [km]'])
df_PGA_aumentado['MlEpi'] = df_PGA_aumentado['Ml'] * df_PGA_aumentado['logEpi']
df_PGA_aumentado['MlHip'] = df_PGA_aumentado['Ml'] * df_PGA_aumentado['logHip']

df_PGA_aumentado.describe().round(2)

In [ ]:
df_PGA_aumentado.to_csv('df_PGA_BRJC_SG.csv',
                        sep=';',
                        decimal=',',
                        index=False)

# Plots Parametros

In [ ]:
df_PGA_aumentado['Color Prof'] = 'b'
filtro_ = (df_PGA_aumentado['Profundidad [km]'] > 10) & (df_PGA_aumentado['Profundidad [km]'] <= 20)
df_PGA_aumentado['Color Prof'][filtro_] = 'g'
filtro_ = (df_PGA_aumentado['Profundidad [km]'] > 20) & (df_PGA_aumentado['Profundidad [km]'] <= 40)
df_PGA_aumentado['Color Prof'][filtro_] = 'y'
filtro_ = (df_PGA_aumentado['Profundidad [km]'] > 40)
df_PGA_aumentado['Color Prof'][filtro_] = 'r'


In [ ]:
df_PGA_aumentado['Size'] = 2
filtro_ = (df_PGA_aumentado['Ml'] >= 4) & (df_PGA_aumentado['Ml'] < 5)
df_PGA_aumentado['Size'][filtro_] = 5
filtro_ = (df_PGA_aumentado['Ml'] >= 5)
df_PGA_aumentado['Size'][filtro_] = 10


## X-Y-Z

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5,5))

plt.scatter(df_PGA_aumentado['X [km]'],
            df_PGA_aumentado['Y [km]'],
            s= df_PGA_aumentado['Size'] * 10,
            color=df_PGA_aumentado['Color Prof'],
            alpha=0.7)

plt.scatter(df_BRJC['X [km]'],
            df_BRJC['Y [km]'],
            s= 100,
            marker='*',
            color= 'k',
            alpha=1.0)

plt.axis('equal')
plt.grid(ls='--', color='gray')
plt.xlabel('X [km]')
plt.ylabel('Y [km]')

plt.show()

## Dist Epi - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Epi [km]'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlim(1, 200)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Epi. [km]')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Epi [km]'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')

# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)
ax.set_xlim(1, 200)
# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Epi. [km]')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Epi [km]'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1, 200)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Epi. [km]')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## Dist Hip - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Hip [km]'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(10, 200)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Hip. [km]')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Hip [km]'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(10, 200)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Hip. [km]')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Dist Hip [km]'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(10, 200)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Dist. Hip. [km]')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## Profundidad - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Profundidad [km]'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(90, 320)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Profundidad [km]')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Profundidad [km]'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(90, 320)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Profundidad [km]')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Profundidad [km]'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(90, 320)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Profundidad [km]')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## Ml - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Ml'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0, 5.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Ml'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0, 5.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Ml'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_yscale('log')
ax.set_xlim(0, 5.0)
# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(np.sqrt(df_PGA_aumentado['Ml']),
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)
# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('sqrt(Ml)')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## Rumbo - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Rumbo [°]'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Rumbo [°]')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Rumbo [°]'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Rumbo [°]')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['Rumbo [°]'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)
# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Rumbo [°]')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5,5),
                       subplot_kw={"projection":"polar"})

ax.scatter(np.deg2rad(df_PGA_aumentado['Rumbo [°]']),
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

# Norte arriba y sentido horario
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)

# Escala logarítmica radial
ax.set_rscale("log")

plt.suptitle('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5,5),
                       subplot_kw={"projection":"polar"})

ax.scatter(np.deg2rad(df_PGA_aumentado['Rumbo [°]']),
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

# Norte arriba y sentido horario
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)

# Escala logarítmica radial
ax.set_rscale("log")

plt.suptitle('PGA NS [$cm/s^2$]')

plt.show()


In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5,5),
                       subplot_kw={"projection":"polar"})

ax.scatter(np.deg2rad(df_PGA_aumentado['Rumbo [°]']),
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

# Norte arriba y sentido horario
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)

# Escala logarítmica radial
ax.set_rscale("log")

plt.suptitle('PGA Z [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=2, figsize=(7,8))

ax[0].scatter(df_PGA_aumentado['Seno'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[0].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[0].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[0].minorticks_on()

# Grilla mayor
ax[0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0].set_xlabel('Seno')
ax[0].set_ylabel('PGA EW [$cm/s^2$]')

########
# Coseno
ax[1].scatter(df_PGA_aumentado['Coseno'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[1].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[1].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[1].minorticks_on()

# Grilla mayor
ax[1].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[1].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[1].set_xlabel('Coseno')
ax[1].set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=2, figsize=(7,8))

ax[0].scatter(df_PGA_aumentado['Seno'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[0].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[0].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[0].minorticks_on()

# Grilla mayor
ax[0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0].set_xlabel('Seno')
ax[0].set_ylabel('PGA NS [$cm/s^2$]')

########
# Coseno
ax[1].scatter(df_PGA_aumentado['Coseno'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[1].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[1].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[1].minorticks_on()

# Grilla mayor
ax[1].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[1].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[1].set_xlabel('Coseno')
ax[1].set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=2, figsize=(7,8))

ax[0].scatter(df_PGA_aumentado['Seno'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[0].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[0].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[0].minorticks_on()

# Grilla mayor
ax[0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0].set_xlabel('Seno')
ax[0].set_ylabel('PGA Z [$cm/s^2$]')

########
# Coseno
ax[1].scatter(df_PGA_aumentado['Coseno'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax[1].set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax[1].yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax[1].minorticks_on()

# Grilla mayor
ax[1].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[1].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[1].set_xlabel('Coseno')
ax[1].set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## MlEpi - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlEpi'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlEpi'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlEpi'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

## MlHip - PGA

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlHip'],
           df_PGA_aumentado['PGA EW (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA EW [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlHip'],
           df_PGA_aumentado['PGA NS (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA NS [$cm/s^2$]')

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

ax.scatter(df_PGA_aumentado['MlHip'],
           df_PGA_aumentado['PGA Z (cm/s^2)'],
           s= df_PGA_aumentado['Size'] * 10,
           color=df_PGA_aumentado['Color Prof'],
           alpha=0.7)

#ax.set_xscale('log')
ax.set_yscale('log')
#ax.set_xlim(3.0, 7.0)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Ml * Log10(Dist. Epi.)')
ax.set_ylabel('PGA Z [$cm/s^2$]')

plt.show()

# Modelos PGA empiricos Notebook 09

In [ ]:
def PGA_EW_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 4.7827
  a = 0.6840
  b = -3.5873
  c = 0.0048
  sigma = 0.2734

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

def PGA_NS_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 5.0227
  a = 0.6720
  b = -3.6422
  c = 0.0054
  sigma = 0.2594

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

def PGA_Z_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 4.2114
  a = 0.6723
  b = -3.4934
  c = 0.0053
  sigma = 0.2324

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

In [ ]:
df_PGA_aumentado.columns

In [ ]:
df_PGA_aumentado['Mw'] = df_PGA_aumentado['Ml'] * 0.958 + 0.1

df_PGA_aumentado['PGA_EW_cal'] = PGA_EW_m03(Mw = df_PGA_aumentado['Mw'],
                                            Repi= df_PGA_aumentado['Dist Epi [km]'],
                                            H= df_PGA_aumentado['Profundidad [km]'],
                                            X= 0)
df_PGA_aumentado['Residual_EW'] = np.log10(df_PGA_aumentado['PGA EW (cm/s^2)']) - df_PGA_aumentado['PGA_EW_cal']
df_PGA_aumentado['PGA_EW_cal'] = 10**df_PGA_aumentado['PGA_EW_cal']

df_PGA_aumentado['PGA_NS_cal'] = PGA_NS_m03(Mw = df_PGA_aumentado['Mw'],
                                            Repi= df_PGA_aumentado['Dist Epi [km]'],
                                            H= df_PGA_aumentado['Profundidad [km]'],
                                            X= 0)
df_PGA_aumentado['Residual_NS'] = np.log10(df_PGA_aumentado['PGA NS (cm/s^2)']) - df_PGA_aumentado['PGA_NS_cal']
df_PGA_aumentado['PGA_NS_cal'] = 10**df_PGA_aumentado['PGA_NS_cal']

df_PGA_aumentado['PGA_Z_cal'] = PGA_Z_m03(Mw = df_PGA_aumentado['Mw'],
                                          Repi= df_PGA_aumentado['Dist Epi [km]'],
                                          H= df_PGA_aumentado['Profundidad [km]'],
                                          X= 0)
df_PGA_aumentado['Residual_Z'] = np.log10(df_PGA_aumentado['PGA Z (cm/s^2)']) - df_PGA_aumentado['PGA_Z_cal']
df_PGA_aumentado['PGA_Z_cal'] = 10**df_PGA_aumentado['PGA_Z_cal']

## Plots

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA EW (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_EW_cal'].min())

xmax = max(df_PGA_aumentado['PGA EW (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_EW_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA EW (cm/s^2)'],
           df_PGA_aumentado['PGA_EW_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_EW'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_EW'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_EW"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_EW'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_EW'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA EW [$cm/s^2$]')

plt.tight_layout()

plt.show()

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA NS (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_NS_cal'].min())

xmax = max(df_PGA_aumentado['PGA NS (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_NS_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA NS (cm/s^2)'],
           df_PGA_aumentado['PGA_NS_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_NS'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_NS'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_NS"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_NS'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_NS'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA NS [$cm/s^2$]')

plt.tight_layout()

plt.show()

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA Z (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_Z_cal'].min())

xmax = max(df_PGA_aumentado['PGA Z (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_Z_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA Z (cm/s^2)'],
           df_PGA_aumentado['PGA_Z_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_Z'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_Z'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_Z"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_Z'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_Z'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA Z [$cm/s^2$]')

plt.tight_layout()

plt.show()

# Modelos PGA nuevos

In [ ]:
df_PGA_aumentado['logPGA_EW'] = np.log10(df_PGA_aumentado['PGA EW (cm/s^2)'])
df_PGA_aumentado['logPGA_NS'] = np.log10(df_PGA_aumentado['PGA NS (cm/s^2)'])
df_PGA_aumentado['logPGA_Z'] = np.log10(df_PGA_aumentado['PGA Z (cm/s^2)'])

df_X = df_PGA_aumentado[['Ml', 'logEpi', 'logHip',
                         'Profundidad [km]', 'MlEpi', 'MlHip']].copy()

df_y = df_PGA_aumentado[['logPGA_EW', 'logPGA_NS', 'logPGA_Z']].copy()

In [ ]:
df_X.describe()

In [ ]:
df_y.describe()

## Modelo 1

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi"]])

modelo_1_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_1_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_1_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_1_EW.rsquared,
        modelo_1_EW.rsquared_adj,
        sigma,
        modelo_1_EW.aic,
        modelo_1_EW.bic,
        modelo_1_EW.fvalue,
        modelo_1_EW.f_pvalue,
        int(modelo_1_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_1_EW.xlsx"),
                 index=False)

tabla = modelo_1_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_1_EW.xlsx"))


### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi"]])

modelo_1_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_1_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_1_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_1_NS.rsquared,
        modelo_1_NS.rsquared_adj,
        sigma,
        modelo_1_NS.aic,
        modelo_1_NS.bic,
        modelo_1_NS.fvalue,
        modelo_1_NS.f_pvalue,
        int(modelo_1_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_1_NS.xlsx"),
                 index=False)

tabla = modelo_1_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_1_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi"]])

modelo_1_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_1_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_1_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_1_Z.rsquared,
        modelo_1_Z.rsquared_adj,
        sigma,
        modelo_1_Z.aic,
        modelo_1_Z.bic,
        modelo_1_Z.fvalue,
        modelo_1_Z.f_pvalue,
        int(modelo_1_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_1_Z.xlsx"),
                 index=False)

tabla = modelo_1_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_1_Z.xlsx"))

## Modelo 2

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip"]])

modelo_2_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_2_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_2_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_2_EW.rsquared,
        modelo_2_EW.rsquared_adj,
        sigma,
        modelo_2_EW.aic,
        modelo_2_EW.bic,
        modelo_2_EW.fvalue,
        modelo_2_EW.f_pvalue,
        int(modelo_2_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_2_EW.xlsx"),
                 index=False)

tabla = modelo_2_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_2_EW.xlsx"))

### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip"]])

modelo_2_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_2_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_2_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_2_NS.rsquared,
        modelo_2_NS.rsquared_adj,
        sigma,
        modelo_2_NS.aic,
        modelo_2_NS.bic,
        modelo_2_NS.fvalue,
        modelo_2_NS.f_pvalue,
        int(modelo_2_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_2_NS.xlsx"),
                 index=False)

tabla = modelo_2_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_2_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip"]])

modelo_2_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_2_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_2_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_2_Z.rsquared,
        modelo_2_Z.rsquared_adj,
        sigma,
        modelo_2_Z.aic,
        modelo_2_Z.bic,
        modelo_2_Z.fvalue,
        modelo_2_Z.f_pvalue,
        int(modelo_2_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_2_Z.xlsx"),
                 index=False)

tabla = modelo_2_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_2_Z.xlsx"))

## Modelo 3

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]']])

modelo_3_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_3_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_3_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_3_EW.rsquared,
        modelo_3_EW.rsquared_adj,
        sigma,
        modelo_3_EW.aic,
        modelo_3_EW.bic,
        modelo_3_EW.fvalue,
        modelo_3_EW.f_pvalue,
        int(modelo_3_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_3_EW.xlsx"),
                 index=False)

tabla = modelo_3_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_3_EW.xlsx"))

### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]']])

modelo_3_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_3_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_3_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_3_NS.rsquared,
        modelo_3_NS.rsquared_adj,
        sigma,
        modelo_3_NS.aic,
        modelo_3_NS.bic,
        modelo_3_NS.fvalue,
        modelo_3_NS.f_pvalue,
        int(modelo_3_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_3_NS.xlsx"),
                 index=False)

tabla = modelo_3_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_3_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]']])

modelo_3_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_3_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_3_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_3_Z.rsquared,
        modelo_3_Z.rsquared_adj,
        sigma,
        modelo_3_Z.aic,
        modelo_3_Z.bic,
        modelo_3_Z.fvalue,
        modelo_3_Z.f_pvalue,
        int(modelo_3_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_3_Z.xlsx"),
                 index=False)

tabla = modelo_3_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_3_Z.xlsx"))

## Modelo 4

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]', 'MlHip']])

modelo_4_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_4_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_4_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_4_EW.rsquared,
        modelo_4_EW.rsquared_adj,
        sigma,
        modelo_4_EW.aic,
        modelo_4_EW.bic,
        modelo_4_EW.fvalue,
        modelo_4_EW.f_pvalue,
        int(modelo_4_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_4_EW.xlsx"),
                 index=False)

tabla = modelo_4_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_4_EW.xlsx"))

### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]', 'MlHip']])

modelo_4_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_4_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_4_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_4_NS.rsquared,
        modelo_4_NS.rsquared_adj,
        sigma,
        modelo_4_NS.aic,
        modelo_4_NS.bic,
        modelo_4_NS.fvalue,
        modelo_4_NS.f_pvalue,
        int(modelo_4_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_4_NS.xlsx"),
                 index=False)

tabla = modelo_4_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_4_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", 'Profundidad [km]', 'MlHip']])

modelo_4_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_4_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_4_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_4_Z.rsquared,
        modelo_4_Z.rsquared_adj,
        sigma,
        modelo_4_Z.aic,
        modelo_4_Z.bic,
        modelo_4_Z.fvalue,
        modelo_4_Z.f_pvalue,
        int(modelo_4_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_4_Z.xlsx"),
                 index=False)

tabla = modelo_4_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_4_Z.xlsx"))

## Modelo 5

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip", 'Profundidad [km]', 'MlEpi']])

modelo_5_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_5_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_5_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_5_EW.rsquared,
        modelo_5_EW.rsquared_adj,
        sigma,
        modelo_5_EW.aic,
        modelo_5_EW.bic,
        modelo_5_EW.fvalue,
        modelo_5_EW.f_pvalue,
        int(modelo_5_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_5_EW.xlsx"),
                 index=False)

tabla = modelo_5_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_5_EW.xlsx"))

### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip", 'Profundidad [km]', 'MlEpi']])

modelo_5_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_5_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_5_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_5_NS.rsquared,
        modelo_5_NS.rsquared_adj,
        sigma,
        modelo_5_NS.aic,
        modelo_5_NS.bic,
        modelo_5_NS.fvalue,
        modelo_5_NS.f_pvalue,
        int(modelo_5_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_5_NS.xlsx"),
                 index=False)

tabla = modelo_5_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_5_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logHip", 'Profundidad [km]', 'MlEpi']])

modelo_5_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_5_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_5_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_5_Z.rsquared,
        modelo_5_Z.rsquared_adj,
        sigma,
        modelo_5_Z.aic,
        modelo_5_Z.bic,
        modelo_5_Z.fvalue,
        modelo_5_Z.f_pvalue,
        int(modelo_5_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_5_Z.xlsx"),
                 index=False)

tabla = modelo_5_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_5_Z.xlsx"))

## Modelo 6

### PGA-EW

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", "logHip", 'Profundidad [km]', 'MlEpi', "MlHip"]])

modelo_6_EW = sm.OLS(df_y["logPGA_EW"], X).fit()

print(modelo_6_EW.summary())

In [ ]:
sigma = np.sqrt(modelo_6_EW.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_6_EW.rsquared,
        modelo_6_EW.rsquared_adj,
        sigma,
        modelo_6_EW.aic,
        modelo_6_EW.bic,
        modelo_6_EW.fvalue,
        modelo_6_EW.f_pvalue,
        int(modelo_6_EW.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_6_EW.xlsx"),
                 index=False)

tabla = modelo_6_EW.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_6_EW.xlsx"))

### PGA-NS

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", "logHip", 'Profundidad [km]', 'MlEpi', "MlHip"]])

modelo_6_NS = sm.OLS(df_y["logPGA_NS"], X).fit()

print(modelo_6_NS.summary())

In [ ]:
sigma = np.sqrt(modelo_6_NS.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_6_NS.rsquared,
        modelo_6_NS.rsquared_adj,
        sigma,
        modelo_6_NS.aic,
        modelo_6_NS.bic,
        modelo_6_NS.fvalue,
        modelo_6_NS.f_pvalue,
        int(modelo_6_NS.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_6_NS.xlsx"),
                 index=False)

tabla = modelo_6_NS.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_6_NS.xlsx"))

### PGA-Z

In [ ]:
X = sm.add_constant(df_X[["Ml", "logEpi", "logHip", 'Profundidad [km]', 'MlEpi', "MlHip"]])

modelo_6_Z = sm.OLS(df_y["logPGA_Z"], X).fit()

print(modelo_6_Z.summary())

In [ ]:
sigma = np.sqrt(modelo_6_Z.mse_resid)

print("Sigma residual:", sigma, "unidades log10")

In [ ]:
resumen = pd.DataFrame({
    "Indicador": [
        "R²",
        "R² ajustado",
        "Sigma residual (log10)",
        "AIC",
        "BIC",
        "F",
        "Prob(F)",
        "Observaciones"
    ],
    "Valor": [
        modelo_6_Z.rsquared,
        modelo_6_Z.rsquared_adj,
        sigma,
        modelo_6_Z.aic,
        modelo_6_Z.bic,
        modelo_6_Z.fvalue,
        modelo_6_Z.f_pvalue,
        int(modelo_6_Z.nobs)
    ]
})

resumen.to_excel(os.path.join(path_save_figures,"Resumen_modelo_6_Z.xlsx"),
                 index=False)

tabla = modelo_6_Z.summary2().tables[1]
tabla.to_excel(os.path.join(path_save_figures,
                            "Tabla_modelo_6_Z.xlsx"))

# Fin